### 在 10 个 demo reports 上尝试提取，所有均为doc文件

In [9]:
from docx import Document

def read_docx(file_path):
    doc = Document(file_path)
    full_text = []
    for para in doc.paragraphs:
        full_text.append(para.text)

    return full_text

In [12]:
def extract_tables(file_path):
    # 打开一个 DOCX 文件
    doc = Document(file_path)

    # 用于存储表格数据
    table_data = []

    # 遍历文档中的所有表格
    for table in doc.tables:
        # 用于存储当前表格的数据
        table_content = []
        for row in table.rows:
            row_data = []
            for cell in row.cells:
                # 提取每个单元格中的文本
                row_data.append(cell.text.strip())
            table_content.append(row_data)
        table_data.append(table_content)

    return table_data

In [10]:
from pathlib import Path

root = Path('/Users/jindarui/Desktop/Hetairos_project/data_exchange/demo_reports_felix/converted_docx')

file = root / 'report 1.docx'
output = read_docx(file)

In [13]:
tables = extract_tables(file)

In [14]:
tables

[[['Classification „brain tumor classifier“',
   'Classification „brain tumor classifier“',
   'Methylation class astrocytoma, IDH-mutant, high grade',
   'Methylation class astrocytoma, IDH-mutant, high grade'],
  ['Version „brain tumor classifier“',
   'Version „brain tumor classifier“',
   'V12.5',
   'Score: 0.91'],
  ['', '', '', ''],
  ['Copy number variations:',
   'segmental loss of Chromosome 1q, gain of Chromosome 6p, gain of Chromosome 7 (incl. EGFR, CDK6, MET), gain of Chromosome 8 (incl. MYC and MYBL1), gain of Chromosome 10p, segmental loss of Chromosome 10q, loss of Chromosome 11p, segmental loss of Chromosome 13q (incl. RB1), fokal loss of Chromosome 16q',
   'segmental loss of Chromosome 1q, gain of Chromosome 6p, gain of Chromosome 7 (incl. EGFR, CDK6, MET), gain of Chromosome 8 (incl. MYC and MYBL1), gain of Chromosome 10p, segmental loss of Chromosome 10q, loss of Chromosome 11p, segmental loss of Chromosome 13q (incl. RB1), fokal loss of Chromosome 16q',
   'segmen

In [11]:
output

['Analysis:',
 'Panel-Sequencing, 850k-Analysis',
 '',
 'Clinical Data:',
 'Supplier: Working group Glioblastoma, Milan Italy. Suppliers diagnosis: Astrocytoma, IDH-mutant, WHO Grade 4. Localization: frontal left. Request of Panel-Sequencing and 850k-Methylationassay. MGMT-Pyrosequencing methylated, no TERT-Promoter-mutation, no homozygous deletion of CDKN2A/B, no amplification of EGFR-gen, Ki 67 3-5%, Mitoses 4 in 10 „high power fields“, evidence of coagulative necrosis and vascular proliferation.',
 '',
 'Material:',
 'We received 1 paraffin block with the external Number 22-1-22790 (= our N/2022/2957). Tumor cell content 60%. Internal ID tumor-DNA 320538.',
 '',
 'Method:',
 'Analysis using Illumina Human Methylation 850 (850k) Array and internal classifier V11b4.',
 '',
 'Mutational analysis on tumor DNA via “Next Generation Sequencing” on Illumina NextSeq platform for 201 brain tumor-related genes applying the panel NPHD2022A, enriched with Agilent SureSelect method. To filter fro

### 通过 ollama 调用本地llm 对文本进行处理

In [ ]:
import lmstudio as lms

# load the model from lmstudio 
model = lms.llm("google/gemma-3n-e4b")

In [ ]:
from pathlib import Path
from tqdm import tqdm
import pdfplumber
import pandas as pd

# Define the path to the text files
files_root = 'txt/'
files = list(Path(files_root).glob('*'))
files.sort()

def convert_txt(file):
    with pdfplumber.open(file) as pdf:
        text = ''
        for page in pdf.pages:
            text += page.extract_text() or ''
    return text

info_df = pd.DataFrame(columns=[
    "Patho ID",
    "Entry date",
    "Sex",
    "Date of birth",
    "Einsender",
    "Einsenderdiagnose",
    "Histological diagnosis",
    "Molecular diagnosis",
    "Integrated diagnosis",
    "Methylation class",
    "Methylation score",
    "Immunohistochemistry",
    "Microscopy results"
])

for idx, file in tqdm(enumerate(files), total=len(files), desc="Processing files", colour='green', ncols=100):
    if file.suffix == '.pdf':
        report_text = convert_txt(file)
    elif file.suffix == '.txt':
        with open(file, 'r', encoding='utf-8') as f:
            report_text = f.read()
    else:
        print(f"Unsupported file type: {file}")
        continue

    results = model.respond(
        f"""请从以下内容中提取病例的
        "Patho ID" (N/XX/XXXXXX);
        "Entry date" (DD/MM/YYYY);
        "Sex" (F/M);
        "Date of birth" (DD/MM/YYYY);
        "Einsender";
        "Einsenderdiagnose";
        "Histological diagnosis";
        "Molecular diagnosis";
        "Integrated diagnosis";
        "Methylation class";
        "Methylation score";
        "Immunohistochemistry";
        "Microscopy results".
        只需要输出以上内容，如果没有则返回 na: {report_text} /no_think""")
    
    results = str(results)
    results_dict = {}
    for line in results.split('\n'):
        if ':' in line:
            key, value = line.split(':', 1)
            results_dict[key.strip()] = value.strip()
    
    # Fill missing keys with 'na'
    for key in info_df.columns:
        if key not in results_dict:
            results_dict[key] = 'na'
    
    # Append the results to the DataFrame
    info_df.loc[info_df.shape[0]] = results_dict
    info_df.to_excel('./info_df_gemma4b_0.9.xlsx', index=False)

In [14]:
import requests
import sys
from pathlib import Path
from tqdm import tqdm
import pdfplumber
import pandas as pd

OLLAMA_API_URL = "http://127.0.0.1:11434/api/chat"
MODEL_NAME = "qwen3:4b"  
TEMPERATURE = 0.8

def convert_txt(file):
    with pdfplumber.open(file) as pdf:
        text = ''
        for page in pdf.pages:
            text += page.extract_text() or ''
    return text
    
info_df = pd.DataFrame(columns=[
    "Patho ID",
    "Entry date",
    "Sex",
    "Date of birth",
    "Einsender",
    "Einsenderdiagnose",
    "Histological diagnosis",
    "Molecular diagnosis",
    "Integrated diagnosis",
    "Methylation class",
    "Methylation score",
    "Immunohistochemistry",
    "Microscopy results"
])

files_root = '/Users/jindarui/Desktop/Hetairos_project/data_exchange/demo_reports_felix/txt/'
files = list(Path(files_root).glob('*'))
files.sort()

for idx, file in tqdm(enumerate(files), total=len(files), desc="Processing files", colour='green', ncols=100):
    if file.suffix == '.pdf':
        report_text = convert_txt(file)
    elif file.suffix == '.txt':
        with open(file, 'r', encoding='utf-8') as f:
            report_text = f.read()
    else:
        print(f"Unsupported file type: {file}")
        continue
    
    prompt = f"""请从以下内容中提取病例的
            "Patho ID" (N/XX/XXXXXX);
            "Entry date" (DD/MM/YYYY);
            "Sex" (F/M);
            "Date of birth" (DD/MM/YYYY);
            "Einsender";
            "Einsenderdiagnose";
            "Histological diagnosis";
            "Molecular diagnosis";
            "Integrated diagnosis";
            "Methylation class";
            "Methylation score";
            "Immunohistochemistry";
            "Microscopy results".
            只需要输出以上内容，如果没有则返回 na: {report_text} /no_think"""

    payload = {
        "model": MODEL_NAME,
        "temperature": TEMPERATURE,
        "stream": False,
        "messages": [{
            "role": "user",
            "content": prompt
        }]
    }
    response = requests.post(OLLAMA_API_URL, json=payload)
    if response.status_code == 200:
        results = response.json()["message"]["content"]   
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        sys.exit(1)

    results = str(results)
    results_dict = {}
    for line in results.split('\n'):
        if ':' in line:
            key, value = line.split(':', 1)
            results_dict[key.strip()] = value.strip()
    
    # Fill missing keys with 'na'
    for key in info_df.columns:
        if key not in results_dict:
            results_dict[key] = 'na'

    # Append the results to the DataFrame
    info_df.loc[info_df.shape[0]] = results_dict
    info_df.to_excel('./info_df_qwen4b_0.8.xlsx', index=False)

Processing files: 100%|████████████████████████████████████████████| 10/10 [40:56<00:00, 245.63s/it]


### test langextract

In [16]:
import langextract as lx

In [17]:
files_root = 'txt/'
files = list(Path(files_root).glob('*'))
files.sort()

file = files[0]

with open(file, 'r', encoding='utf-8') as f:
    report_text = f.read()

prompt = f"""please extract the following information from the text:
            "Patho ID" (N/XX/XXXXXX);
            "Entry date" (DD/MM/YYYY);
            "Sex" (F/M);
            "Date of birth" (DD/MM/YYYY);
            "Einsender";
            "Einsenderdiagnose";
            "Histological diagnosis";
            "Molecular diagnosis";
            "Integrated diagnosis";
            "Methylation class";
            "Methylation score";
            "Immunohistochemistry";
            "Microscopy results".
            you only need to output the above content, if not available, return 'na'."""

examples = [
    lx.data.ExampleData(
        text=(
            report_text
            ),
        extractions=[
            lx.data.Extraction(
                extraction_class="Patho ID",
                extraction_text="N/22/002957",
            ),
            lx.data.Extraction(
                extraction_class="Entry date",
                extraction_text="28.11.2022",
            ),
            lx.data.Extraction(
                extraction_class="Sex",
                extraction_text="F",
            ),
            lx.data.Extraction(
                extraction_class="Date of birth",
                extraction_text="01.02.1967",
            ),
            lx.data.Extraction(
                extraction_class="Einsender",
                extraction_text="Working group Glioblastoma, Milan Italy",
            ),
            lx.data.Extraction(
                extraction_class="Einsenderdiagnose",
                extraction_text="Astrocytoma, IDH-mutant, WHO Grade 4",
            ),
            lx.data.Extraction(
                extraction_class="Histological diagnosis",
                extraction_text="Astrocytoma IDH-mutant",
            ),
            lx.data.Extraction(
                extraction_class="Molecular diagnosis",
                extraction_text="Methylation class „astrocytoma, IDH-mutant, high grade“, ATRX-frameshift-mutation, IDH1 R132H-mutation and TP53-mutation, MGMT methylated",
            ),
            lx.data.Extraction(
                extraction_class="Integrated diagnosis",
                extraction_text="Astrocytoma, IDH-mutant, WHO Grade 3",
            ),
            lx.data.Extraction(
                extraction_class="Methylation class",
                extraction_text="Astrocytoma, Idh Mutant; High Grade",
            ),
            lx.data.Extraction(
                extraction_class="Methylation score",
                extraction_text="0.91",
            ),
            lx.data.Extraction(
                extraction_class="Immunohistochemistry",
                extraction_text="GFAP: tumor cells positive; Ki 67: up to 5%; ERG: endothelial cells positive",
            ),
            lx.data.Extraction(
                extraction_class="Microscopy results",
                extraction_text="histologically a diffuse astrocytoma with protoplasmic appearance, few mitoses and low proliferation index of up to 5%. The histological criteria for grade 4 are not convincingly evident in the fragment received.",
            )
        ],
    )
]

with open(files[1], 'r', encoding='utf-8') as f:
    report_text_ = f.read()
    
result = lx.extract(
    text_or_documents=report_text_,
    prompt_description=prompt,
    examples=examples,
    model_id="gemma3n:e4b",  # Automatically selects Ollama provider
    model_url="http://localhost:11434",
    fence_output=False,
    use_schema_constraints=False
)

IndexError: list index out of range